# Give the proposer feedback

<a id="supply-feedback"></a>

A score tells you how well a revision performed. [Evidence](https://sentient-xyz.github.io/meta-evolve-docs/concepts/evidence/)
such as failed checks tells you what needs fixing. This guide records those
checks, supplies them to a proposer,
and uses them to choose the next revision. Then you will turn feedback off
and compare the results.

| You want to… | Use |
|---|---|
| [Record failed checks](#record-what-failed) | `EvaluationResult` with evidence |
| [Use checks in a revision](#put-authorized-history-in-the-message) | The proposer's `context` argument |
| [Supply prior observations](#declare-the-feedback-policy) | `RecentAncestors` |
| [Compare with feedback off](#change-and-predict) | `context=None` |

The proposer chooses handwritten revisions; Python executes and grades them.
No model, SDK, or credentials are needed.

Here, **feedback** means evaluator observations supplied to the proposer.
The API's broader **experience** vocabulary also covers other authorized
history and [explicit pull access](https://sentient-xyz.github.io/meta-evolve-docs/concepts/experience/).



<a id="set-up-this-page"></a>
<a id="complete-shared-source"></a>
<a id="1-install"></a>
<a id="2-define-the-parser-and-its-checks"></a>

## Required setup for a fresh notebook

Use a fresh notebook environment running **Python 3.12 or newer**.
Install directly from the published documentation:

In [ ]:
%pip install https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-evolve.zip

If you already imported Meta-Evolve, restart the kernel after installing.
Then run the remaining cells in order.

**Archived or offline docs:** use the ZIP included with that build. Put
`meta-evolve.zip` in the notebook's working folder (`%pwd` shows it; hosted
notebooks let you upload files), then run `%pip install ./meta-evolve.zip`
instead. Installing from source may still download build tools.

**Starting source and instructions.** `SEED` reads a duration's number but
ignores its unit. `CONTRACT` tells the proposer what the parser should do.

In [ ]:
import meta_evolve as meta

CONTRACT = """Implement parse_seconds(text) for whole-number durations.
Inputs contain a number followed by s, m, or h, such as '30s' or '2h'.
Return the duration in seconds as an integer. Return only Python source.
"""

SEED = '''def parse_seconds(text):
    return int(text[:-1])
'''

**Checks.** `CASES` holds six fixed expected answers. `load_parser` executes
the source locally; use these checks with the reviewed, handwritten code here.

In [ ]:
CASES = (
    ("30s", 30),
    ("90s", 90),
    ("2m", 120),
    ("3m", 180),
    ("1h", 3600),
    ("2h", 7200),
)


def load_parser(source):
    namespace = {}
    exec(source, namespace)
    return namespace["parse_seconds"]

**Revisions.** `MINUTES` fixes minutes; `COMPLETE` adds hours as well.

In [ ]:
MINUTES = '''def parse_seconds(text):
    quantity = int(text[:-1])
    return quantity * 60 if text[-1] == "m" else quantity
'''

COMPLETE = '''def parse_seconds(text):
    quantity = int(text[:-1])
    seconds_per_unit = {"s": 1, "m": 60, "h": 3600}
    return quantity * seconds_per_unit[text[-1]]
'''

The [Start here walkthrough](https://sentient-xyz.github.io/meta-evolve-docs/start-here/) introduces the starting parser.


<a id="record-what-failed"></a>

<a id="3-record-useful-feedback"></a>

## 1. Record useful feedback

A scalar evaluator can return just a score. [`meta.EvaluationResult`][meta_evolve.EvaluationResult]
can also attach **evidence**: observations that explain a measurement.
[`meta.EvidenceDraft`][meta_evolve.EvidenceDraft] describes one observation to
record. We will store each wrong answer's input, expected value, and actual value.

In [ ]:
def evaluate_with_feedback(source):
    parse = load_parser(source)
    failures = []
    for text, expected in CASES:
        actual = parse(text)
        if type(actual) is not int or actual != expected:
            failures.append({
                "input": text, "expected": expected, "actual": actual,
            })

    return meta.EvaluationResult(
        metrics={"score": (len(CASES) - len(failures)) / len(CASES)},
        evidence=(meta.EvidenceDraft(
            kind="failed-cases", data={"failures": failures},
        ),),
    )

`score` is the fraction of checks passed; `1.0` means all six pass.
`"failed-cases"` is our label for this evidence, and `data` holds its contents.
An exception while loading or running the parser still produces a failed
evaluation with no score.

<a id="4-use-the-failed-checks-to-choose-a-revision"></a>

## 2. Use the failed checks to choose a revision

<a id="put-authorized-history-in-the-message"></a>

A proposer requests keyword-only `context`. `context.evidence.latest("failed-cases")`
returns the last matching evidence in the selected bundle, with its contents in
`.data`. Older ancestors may still have failures we have already fixed.

This proposer fixes minutes if any minute checks failed, then hours if any hour
checks failed. If there is no feedback, or nothing failed, it returns the source
unchanged. That rule makes the effect of feedback visible.

In [ ]:
def propose_with_feedback(source, *, context):
    if not context.evidence.enabled:
        return source
    latest = context.evidence.latest("failed-cases")
    if latest is None:
        return source

    failed_inputs = [case["input"] for case in latest.data["failures"]]
    if any(text.endswith("m") for text in failed_inputs):
        return MINUTES
    if any(text.endswith("h") for text in failed_inputs):
        return COMPLETE
    return source

This is a small rule-based demonstration, with two known fixes. The proposer
uses the recorded checks; it does not call the evaluator itself. With a model,
you would put relevant observations into your SDK request. The bundle also
provides `context.bundle.rendered` when you want its text rendering.
The enabled check deliberately makes feedback optional for the comparison
below. Calling `latest()` with feedback disabled instead raises
`meta_evolve.errors.ContextDisabled`. A declared policy with no matching
selected evidence returns `None`. Both proposers and evaluators can record
evidence; this example reserves `"failed-cases"` for its evaluator.

<a id="change-the-feedback-declaration"></a>
<a id="declare-the-feedback-policy"></a>

<a id="5-supply-feedback-and-run"></a>

## 3. Supply feedback and run

Recording evidence does not automatically send it to the proposer.
[`meta.RecentAncestors`][meta_evolve.RecentAncestors] selects recent prior records
from the current parent's lineage. Rejected siblings are
outside that ancestry. `max_records` limits the structured records;
`max_chars` limits their text rendering. Selected records retain canonical order.
A small record limit can still omit evidence; latest means latest in this
bundle, not necessarily the current parent's feedback. `AncestorsOnly` remains
available when you want oldest-record retention instead.

For one bounded run, pass an objective and the policy directly to `improve()`:

In [ ]:
simple_feedback = meta.improve(
    seed=SEED, proposer=propose_with_feedback, evaluator=evaluate_with_feedback,
    objective=meta.Maximize("score", satisfy=1.0),
    trials=2, context=meta.RecentAncestors(max_records=40, max_chars=16_000),
)

Run the equivalent expanded declaration below; we will reuse it to compare
feedback on and off.

In [ ]:
feedback_experiment = meta.Experiment(
    seed=SEED,
    proposer=propose_with_feedback,
    task=meta.Task(evaluator=evaluate_with_feedback,
                   objectives=(meta.Maximize("score", satisfy=1.0),)),
    search=meta.Greedy(max_trials=2),
    context=meta.RecentAncestors(max_records=40, max_chars=16_000),
)
feedback_result = meta.run(feedback_experiment)

<a id="run-and-inspect"></a>
<a id="inspect-the-observations"></a>

<a id="6-read-the-checks-and-the-result"></a>

## 4. Read the checks and the result

`trials()` lists the seed followed by each attempted revision.
Each entry's `evidence` contains its recorded observations. The proposal count
in `usage().trials` excludes the seed; `usage().evaluations` includes its
evaluation. These runs make two proposals and three evaluations.

In [ ]:
first_attempt = feedback_result.trials()[0]
for case in first_attempt.evidence[0].data["failures"]:
    print(f"{case['input']!r}: expected {case['expected']}, got {case['actual']}")
# Output:
# '2m': expected 120, got 2
# '3m': expected 180, got 3
# '1h': expected 3600, got 1
# '2h': expected 7200, got 2

The seed's minute failures lead to `MINUTES`. Its remaining hour failures lead
to `COMPLETE`. `best_trial()` returns the selected attempt and its measurements.

In [ ]:
for number, attempt in enumerate(feedback_result.trials()):
    failures = attempt.evidence[0].data["failures"]
    print(f"Version {number}: {attempt.metrics['score']:.0%}; "
          f"failed checks: {len(failures)}")
print(f"Selected score: {feedback_result.best_trial().metrics['score']:.0%}")
# Output:
# Version 0: 33%; failed checks: 4
# Version 1: 67%; failed checks: 2
# Version 2: 100%; failed checks: 0
# Selected score: 100%

The evaluator and cases stayed fixed. The proposer used observations to choose
source, and independent evaluation determined whether that choice helped.

## Change and predict

Turn feedback off while keeping the same task, seed, proposer, and search.
Python's `replace` copies a declaration with only the named field changed.
Predict the score before running:

In [ ]:
from dataclasses import replace

without_feedback = meta.run(replace(feedback_experiment, context=None))
print(f"With feedback: {feedback_result.best_trial().metrics['score']:.0%}")
print(f"Without feedback: {without_feedback.best_trial().metrics['score']:.0%}")
print("Evaluations in each run:", feedback_result.usage().evaluations,
      without_feedback.usage().evaluations)
# Output:
# With feedback: 100%
# Without feedback: 33%
# Evaluations in each run: 3 3

Without supplied observations, this proposer returns the seed unchanged twice.
The evaluator still records failed checks, but the proposer does not receive
them. Both runs evaluate the seed and two proposals. The score difference
comes from the explicit rule above; it is not a prediction of model performance
or accuracy beyond these six development checks.

## Why isn't my proposer receiving feedback?

Recording evidence and supplying it are separate steps. Declare a context policy,
then read `context.evidence.latest(kind)` in the proposer. Check `.enabled` for
disabled context and the exact evidence label for an enabled lookup with no
match. `RecentAncestors` favors newer ancestor records; rejected siblings remain
outside that ancestry. In this example the bounds retain the full selected
history, so the latest failed checks reflect the current parent.

For missing or truncated observations, see the
[context bounds reference](https://sentient-xyz.github.io/meta-evolve-docs/guides/experience/#pushed-context).

For your own task, record actionable observations in the evaluator and adapt
the proposer to use them. A model proposer must put the selected observations
into its SDK request. Keep the evaluator and budget fixed while comparing
feedback on and off. The [experience concepts](https://sentient-xyz.github.io/meta-evolve-docs/concepts/experience/)
explain what is shared and who controls access.

<a id="context-content-is-a-different-component"></a>

Next, [inspect results and failures](https://sentient-xyz.github.io/meta-evolve-docs/learn/03-inspect-results/) to trace what the
proposer tried and why a version stayed selected. The
[experience reference](https://sentient-xyz.github.io/meta-evolve-docs/guides/experience/) covers other context policies
and explicit history inspection; the [live parser example](https://sentient-xyz.github.io/meta-evolve-docs/guides/live-parser/)
shows an SDK request containing development observations.